# WavLM 구간 가중치 실험 · NVIDIA L4

**Frozen WavLM → 구간 특징 1회 저장 → LR / 평균 head / 가중치 head 비교 → 설정 잠금 → 결과 ZIP.**

본체 fine-tuning은 하지 않습니다. 작은 분류기만 학습합니다. 기존 통화 평균 cache에는 구간 정보가 없어서 **음성 특징 재추출은 필요**합니다.
학습에 쓰는 원본은 서울 Training 데이터이며, 공식 Validation은 사용하지 않습니다.

### 사용법
1. 이 노트북 하나를 Colab에 업로드하고 런타임을 **L4**로 선택합니다.
2. 아래 셀을 순서대로 실행합니다. 경로는 기존 Drive 결과에서 자동으로 읽습니다.
3. **200통화 benchmark 결과의 예상 시간을 확인한 뒤** 전체 실험 셀을 실행합니다.
4. 마지막 셀에서 결과 ZIP을 다운로드합니다. 실행 후 노트북도 저장해 로그를 보존하세요.

긴 압축 문자열이 없으며 폴더 전체 자동 탐색도 하지 않습니다. 코드 셀은 기본으로 접혀 있습니다.

### 시간·공간
- 첫 준비: 보통 수분. 모델 최초 다운로드 시간은 별도입니다.
- 전체 음성 특징 추출: **시간 단위 작업**입니다. 실제 200통화의 읽기·추론·저장 시간을 측정해 Train/평가 예상 시간을 표시합니다.
- head 학습: 작은 모델이지만 3-fold 내부 early stopping과 재학습을 포함하므로 분~시간 단위입니다. 첫 fold가 끝나면 남은 fold 시간을 추정합니다.
- 여유 Drive 공간 4GiB 이상. L4 실측 전 고정 완료시간이나 99% 달성은 보장하지 않습니다.
- 완료한 64통화 블록, head의 epoch, 완료 fold부터 재개합니다. 중단된 블록/epoch의 미저장 작업은 다시 수행합니다.
- 하나의 출력 폴더에는 한 런타임만 연결하세요.


In [ ]:
#@title 1. Drive 연결 및 경로 확인
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, sys, subprocess
ROOT = Path('/content/drive/MyDrive/DDC-Colab')
INPUT_JSON = str(ROOT/'wavlm_gender_inputs.json')
DATA_ROOT = ''  # 기존 identity에서 자동으로 읽음. 필요하면 Training/의 상위 폴더 입력
OUTPUT = str(ROOT/'wavlm_window_attention_l4_v1')
HF_CACHE = '/content/wavlm_window_hf_cache'
previous = ROOT/'wavlm_gender_adaptation_l4_bf16_v2/identity.json'
if not DATA_ROOT and previous.exists():
    DATA_ROOT = json.loads(previous.read_text())['config']['data_root']
if not DATA_ROOT:
    DATA_ROOT = input('Training/ 폴더를 포함한 데이터 루트 경로: ').strip()
assert Path(INPUT_JSON).is_file(), 'DDC-Colab/wavlm_gender_inputs.json을 확인하세요.'
assert (Path(DATA_ROOT)/'Training').is_dir(), 'DATA_ROOT 바로 아래에 Training/이 있어야 합니다.'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
assert 'L4' in gpu, f'L4 런타임을 선택하세요: {gpu}'
print('GPU:', gpu, '\n데이터:', DATA_ROOT, '\n새 결과:', OUTPUT)
WORK = Path('/content/wavlm_window_attention_v1')
WORK.mkdir(exist_ok=True)
CONFIG = WORK/'config.json'
CONFIG.write_text(json.dumps(dict(input_json=INPUT_JSON, data_root=DATA_ROOT, output=OUTPUT, hf_cache=HF_CACHE), ensure_ascii=False, indent=2))


## 2. 실행 코드 준비
아래 세 셀은 읽을 수 있는 Python 코드를 런타임에 저장합니다. 원본 음성과 이전 결과는 수정하지 않습니다.


In [ ]:
#@title 음성 구간 특징 코드 저장
(WORK/'features.py').write_text(r'''"""Frozen WavLM window features. Identical, label-independent processing for every split."""
import hashlib
import io
import json
import math
import os
from pathlib import Path
import time

import numpy as np

MODEL = 'microsoft/wavlm-large'
REVISION = 'c1423ed94bb01d80a3f5ce5bc39f6026a0f4828c'
INPUT_SHA = 'd3d121bd843d1cfd26cb0bd3a03653d70097ba66f8f412b28f2268f435e1a13a'
JOBS_SHA = '3e4f024ad7c0e5bec8fb2c5ae9af25c467dbc78129d8286faa79a54f237c4e89'
WINDOW = 12
DIM = 1024
BLOCK = 64


def require(ok, message):
    if not ok:
        raise ValueError(message)


def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, allow_nan=False).encode()).hexdigest()


def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()


def read(path):
    return json.loads(Path(path).read_text())


def atomic(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    with temp.open('wb') as f:
        f.write(payload); f.flush(); os.fsync(f.fileno())
    os.replace(temp, path)


def save(path, obj):
    atomic(path, (json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False) + '\n').encode())


def load_jobs(path):
    require(sha(path) == INPUT_SHA, 'Expected the existing wavlm_gender_inputs.json (verified version)')
    source = read(path)
    require(source['format'] == 'wavlm-inputs-v1', 'Input format')
    raw = source['files']['data/jobs.json']
    require(hashlib.sha256(raw.encode()).hexdigest() == JOBS_SHA, 'Job manifest checksum')
    jobs = sorted(json.loads(raw), key=lambda j: j['call_id'])
    require(len(jobs) == len({j['call_id'] for j in jobs}) == 27985, 'Expected 27985 unique calls')
    for partition, count in [('train', 22388), ('internal_validation', 5597)]:
        require(sum(j['partition'] == partition for j in jobs) == count, 'Partition coverage')
    for j in jobs:
        p = Path(j['wav_relative_path'])
        require(not p.is_absolute() and '..' not in p.parts and p.parts[0] == 'Training', 'Official Validation or unsafe audio path')
        require(j['gender'] in ('F', 'M') and j['intervals'], 'Labels/intervals')
    return jobs


def caller_samples(job):
    return sum((round(b * 8000) - round(a * 8000)) * 2 for a, b in job['intervals'])


def waveform(job, root):
    import soundfile as sf
    from scipy.signal import resample_poly
    x, sr = sf.read(Path(root) / job['wav_relative_path'], dtype='float32', always_2d=False)
    require(sr == 8000 and x.ndim == 1 and np.isfinite(x).all(), 'Expected finite 8kHz mono audio')
    parts = []
    for a, b in job['intervals']:
        a, b = round(a * sr), round(b * sr)
        require(0 <= a < b <= len(x), 'Caller annotation outside audio')
        parts.append(resample_poly(x[a:b], 2, 1, window=('kaiser', 5.)).astype(np.float32))
    result = np.concatenate(parts)
    require(len(result) == caller_samples(job), 'Audio coverage mismatch')
    return result


def split_windows(x):
    require(len(x) > 0, 'Empty caller waveform')
    return np.array_split(x, math.ceil(len(x) / (WINDOW * 16000)))


class Encoder:
    def __init__(self):
        import sys
        # Avoid incompatible optional Colab image/audio packages; neither is used here.
        sys.modules['torchvision'] = None
        sys.modules['librosa'] = None
        import torch
        from transformers import AutoFeatureExtractor, WavLMModel
        require(torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(), 'Select Colab NVIDIA L4')
        self.torch = torch
        self.processor = AutoFeatureExtractor.from_pretrained(MODEL, revision=REVISION)
        self.model, loading = WavLMModel.from_pretrained(MODEL, revision=REVISION, output_loading_info=True)
        require(not loading['missing_keys'] and not loading['mismatched_keys'] and not loading['error_msgs'], 'Pretrained weight load incomplete')
        self.model.config.apply_spec_augment = False
        self.model.config.layerdrop = 0.
        self.model.requires_grad_(False).eval().cuda()
        require(self.processor.do_normalize and self.processor.sampling_rate == 16000, 'Processor changed')

    def encode(self, x):
        torch = self.torch
        feats, weights = [], []
        with torch.inference_mode():
            for part in split_windows(x):
                z = self.processor(part, sampling_rate=16000, return_tensors='pt', padding=False).input_values
                if z.shape[1] < 400:
                    z = torch.nn.functional.pad(z, (0, 400 - z.shape[1]))
                with torch.autocast('cuda', dtype=torch.bfloat16):
                    hidden = self.model(z.cuda()).last_hidden_state
                feats.append(hidden[0].float().mean(0).cpu().numpy())
                weights.append(len(part))
        return np.stack(feats).astype(np.float32), np.array(weights, dtype=np.int64)


def array_hash(x, weights, offsets):
    return hashlib.sha256(x.tobytes() + weights.tobytes() + offsets.tobytes()).hexdigest()


def write_block(path, jobs, values, identity):
    x = np.concatenate([a for a, _ in values]).astype(np.float32)
    weights = np.concatenate([b for _, b in values]).astype(np.int64)
    offsets = np.array([0] + list(np.cumsum([len(a) for a, _ in values])), dtype=np.int64)
    buf = io.BytesIO()
    np.savez_compressed(buf, ids=np.array([j['call_id'] for j in jobs]), x=x, weights=weights,
                        offsets=offsets, identity=identity, checksum=array_hash(x, weights, offsets))
    atomic(path, buf.getvalue())


def read_block(path, jobs, identity):
    with np.load(path, allow_pickle=False) as z:
        require(str(z['identity'].item()) == identity, 'Mixed feature identities')
        require(np.array_equal(z['ids'], [j['call_id'] for j in jobs]), 'Feature IDs/order changed')
        x, weights, offsets = z['x'].copy(), z['weights'].copy(), z['offsets'].copy()
        require(str(z['checksum'].item()) == array_hash(x, weights, offsets), 'Feature checksum mismatch')
    require(x.shape == (len(weights), DIM) and x.dtype == np.float32 and np.isfinite(x).all(), 'Invalid feature matrix')
    require(offsets.shape == (len(jobs) + 1,) and offsets[0] == 0 and offsets[-1] == len(x) and (np.diff(offsets) > 0).all(), 'Offsets/coverage')
    require((weights > 0).all() and (weights <= WINDOW * 16000).all(), 'Invalid real-window weights')
    result = []
    for i, j in enumerate(jobs):
        a, b = offsets[i:i+2]
        require(int(weights[a:b].sum()) == caller_samples(j), 'Caller samples missing')
        result.append((x[a:b], weights[a:b]))
    return result


def benchmark_jobs(jobs):
    # Equal coverage over the actual Train duration distribution; no label-specific processing.
    ordered = sorted((j for j in jobs if j['partition'] == 'train'), key=lambda j: (caller_samples(j), j['call_id']))
    return [ordered[i] for i in np.linspace(0, len(ordered)-1, 200, dtype=int)]


def extract(jobs, root, out, identity, partition, encoder=None):
    selected = [j for j in jobs if j['partition'] == partition]
    started = time.monotonic(); measured = []
    for start in range(0, len(selected), BLOCK):
        batch = selected[start:start+BLOCK]
        path = out / 'features' / partition / f'{start:05d}.npz'
        if path.exists():
            read_block(path, batch, identity)
            continue
        t = time.monotonic(); values = []
        for j in batch:
            bp = out / 'benchmark_features' / (j['call_id'] + '.npz')
            if bp.exists():
                values.extend(read_block(bp, [j], identity))
            else:
                if encoder is None:
                    encoder = Encoder()
                values.append(encoder.encode(waveform(j, root)))
        write_block(path, batch, values, identity)
        measured.append((time.monotonic()-t, len(batch)))
        sec = sum(a for a, _ in measured) / sum(b for _, b in measured)
        remaining = len(selected) - start - len(batch)
        print(f'{partition}: {start+len(batch)}/{len(selected)} | ETA {sec*remaining/3600:.2f} h', flush=True)
    save(out / ('extracted_' + partition + '.json'), dict(identity=identity, calls=len(selected), session_seconds=time.monotonic()-started))


def read_partition(jobs, out, identity, partition):
    selected = [j for j in jobs if j['partition'] == partition]; values = []
    for start in range(0, len(selected), BLOCK):
        values.extend(read_block(out / 'features' / partition / f'{start:05d}.npz', selected[start:start+BLOCK], identity))
    return selected, values
''', encoding='utf-8')
print('features.py' + ' 준비 완료')


In [ ]:
#@title 분류기·과적합 점검 코드 저장
(WORK/'heads.py').write_text(r'''"""Small mean/attention heads; outer OOF never participates in early stopping."""
import copy
import io
from pathlib import Path
import warnings
import time

import numpy as np
import torch
from torch import nn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from features import atomic, read, require, save

MAX_EPOCHS = 12
PATIENCE = 3
BATCH = 64
SEED = 42


def pooled(values):
    return np.stack([(x * (w / w.sum())[:, None]).sum(0) for x, w in values]).astype(np.float32)


def metrics(y, p):
    require(p.shape == (len(y), 2) and np.isfinite(p).all(), 'Invalid probabilities')
    require((p >= 0).all() and (p <= 1).all() and np.allclose(p.sum(1), 1, atol=1e-6), 'Probability sum')
    pred = p.argmax(1)
    return dict(n=len(y), correct=int((pred == y).sum()), wrong=int((pred != y).sum()),
                accuracy=float((pred == y).mean()),
                gender_accuracy={g: float((pred[y == i] == i).mean()) for i, g in enumerate('FM')},
                confusion_F_M=[[int(((y == i) & (pred == j)).sum()) for j in range(2)] for i in range(2)])


class Head(nn.Module):
    def __init__(self, kind, dim=1024):
        super().__init__(); self.kind = kind
        # Same seed gives both candidates the exact same initial classifier.
        self.dropout = nn.Dropout(.1)
        self.classifier = nn.Linear(dim, 2)
        if kind == 'attention':
            self.attention = nn.Sequential(nn.Linear(dim, 32), nn.Tanh(), nn.Linear(32, 1, bias=False))
            nn.init.zeros_(self.attention[-1].weight)
        else:
            require(kind == 'mean', 'Unknown head')

    def forward(self, x, weights):
        prior = weights / weights.sum(1, keepdim=True)
        if self.kind == 'attention':
            logits = self.attention(x).squeeze(-1) + prior.clamp_min(1e-20).log()
            logits = logits.masked_fill(weights == 0, -torch.inf)
            alpha = logits.softmax(1)
        else:
            alpha = prior
        pooled_x = (x * alpha.unsqueeze(-1)).sum(1)
        kl = (alpha * (alpha.clamp_min(1e-20).log() - prior.clamp_min(1e-20).log())).sum(1).mean()
        return self.classifier(self.dropout(pooled_x)), kl


def batch(values, ix, mean, scale, device):
    length = max(len(values[i][0]) for i in ix)
    dim = len(mean)
    x = np.zeros((len(ix), length, dim), np.float32)
    w = np.zeros((len(ix), length), np.float32)
    for row, i in enumerate(ix):
        a, b = values[i]; n = len(a)
        x[row, :n] = (a-mean)/scale
        w[row, :n] = b / b.sum()
    return torch.as_tensor(x, device=device), torch.as_tensor(w, device=device)


def torch_save(path, obj):
    b = io.BytesIO(); torch.save(obj, b); atomic(path, b.getvalue())


def load(path):
    # Only load checkpoints generated in this run's identity-locked output.
    return torch.load(path, map_location='cpu', weights_only=False)


def predict_head(record, values, device):
    model = Head(record['kind'], len(record['mean'])).to(device)
    model.load_state_dict(record['state']); model.eval()
    predictions = []
    with torch.inference_mode():
        for start in range(0, len(values), BATCH):
            x, w = batch(values, range(start, min(start+BATCH, len(values))), record['mean'], record['scale'], device)
            predictions.append(model(x, w)[0].softmax(1).cpu().numpy())
    return np.concatenate(predictions)


def train_head(values, y, kind, path, device, stop_values=None, stop_y=None, epochs=None):
    """Tune on an inner stop set; refit from scratch for the selected epoch count."""
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    scaler = StandardScaler().fit(pooled(values))
    mean, scale = scaler.mean_.astype(np.float32), scaler.scale_.astype(np.float32)
    model = Head(kind, len(mean)).to(device)
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=.001)
    limit = epochs if epochs is not None else MAX_EPOCHS
    best_loss = float('inf'); best_epoch = 0; best_state = None; stale = 0; start = 0
    history = []
    if path.exists():
        checkpoint = load(path)
        require(checkpoint['kind'] == kind and checkpoint['limit'] == limit and checkpoint['train_count'] == len(y), 'Head resume mismatch')
        if checkpoint['done']: return checkpoint['record']
        model.load_state_dict(checkpoint['state']); optimizer.load_state_dict(checkpoint['optimizer'])
        best_loss, best_epoch, best_state, stale = (checkpoint[k] for k in ('best_loss', 'best_epoch', 'best_state', 'stale'))
        start, history = checkpoint['epoch'], checkpoint['history']
        torch.set_rng_state(checkpoint['rng'])
        if device.startswith('cuda'): torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
    for epoch in range(start, limit):
        model.train(); order = np.random.default_rng(SEED+epoch).permutation(len(y)); total_loss = 0.
        for begin in range(0, len(order), BATCH):
            ix = order[begin:begin+BATCH]
            x, w = batch(values, ix, mean, scale, device)
            target = torch.as_tensor(y[ix], device=device)
            optimizer.zero_grad(set_to_none=True)
            logits, kl = model(x, w)
            loss = nn.functional.cross_entropy(logits, target) + .01 * kl
            require(torch.isfinite(loss).item(), 'Non-finite head loss')
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.); optimizer.step()
            total_loss += loss.item() * len(ix)
        state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        record = dict(kind=kind, state=state, mean=mean, scale=scale, epochs=epoch+1)
        if stop_values is not None:
            p = predict_head(record, stop_values, device)
            val_loss = float(-np.log(p[np.arange(len(stop_y)), stop_y].clip(1e-12, 1)).mean())
            if val_loss < best_loss - 1e-4:
                best_loss, best_epoch, best_state, stale = val_loss, epoch+1, state, 0
            else:
                stale += 1
        else:
            val_loss = None; best_epoch, best_state = epoch+1, state
        history.append(dict(epoch=epoch+1, training_loss=total_loss/len(y), inner_stop_loss=val_loss))
        done = epoch+1 == limit or (stop_values is not None and stale >= PATIENCE)
        record.update(state=best_state, epochs=best_epoch, history=history)
        torch_save(path, dict(kind=kind, limit=limit, train_count=len(y), epoch=epoch+1,
                             state=state, optimizer=optimizer.state_dict(), best_state=best_state,
                             best_loss=best_loss, best_epoch=best_epoch, stale=stale, history=history,
                             rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if device.startswith('cuda') else [],
                             done=done, record=record))
        print(f'{path.stem}: epoch {epoch+1}/{limit}, train loss {total_loss/len(y):.4f}, stop {val_loss}', flush=True)
        if done: break
    return record


def fit_lr(values, y):
    x = pooled(values); scaler = StandardScaler().fit(x)
    with warnings.catch_warnings(), threadpool_limits(limits=2):
        warnings.simplefilter('error', ConvergenceWarning)
        lr = LogisticRegression(C=.1, max_iter=2000, random_state=SEED).fit(scaler.transform(x), y)
    require(list(lr.classes_) == [0, 1], 'Class order')
    return dict(kind='lr', mean=scaler.mean_, scale=scaler.scale_, coef=lr.coef_, intercept=lr.intercept_)


def predict(record, values, device):
    if record['kind'] != 'lr': return predict_head(record, values, device)
    from scipy.special import expit
    x = (pooled(values)-record['mean'])/record['scale']
    male = expit((x @ record['coef'].T + record['intercept']).ravel())
    return np.column_stack([1-male, male])


def split_plan(y):
    result = []
    for k, (train, test) in enumerate(StratifiedKFold(3, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y)):
        fit, stop = train_test_split(train, test_size=.15, stratify=y[train], random_state=SEED+k)
        require(not set(fit) & set(stop) and not set(train) & set(test), 'Split leakage')
        result.append((train, test, fit, stop))
    return result


def select(y, predictions, folds):
    baseline = metrics(y, predictions['lr']); rows = []
    for kind in ('mean', 'attention'):
        p = predictions[kind]; m = metrics(y, p)
        deltas = [metrics(y[folds == k], p[folds == k])['accuracy'] - metrics(y[folds == k], predictions['lr'][folds == k])['accuracy'] for k in range(3)]
        eligible = min(deltas) >= 0 and m['accuracy'] >= baseline['accuracy'] + .001
        if kind == 'attention':
            # Require a benefit over the matched mean head before attributing benefit to weighting.
            eligible = eligible and m['correct'] > metrics(y, predictions['mean'])['correct']
        rows.append(dict(kind=kind, metrics=m, fold_deltas=deltas, eligible=bool(eligible)))
    good = [r for r in rows if r['eligible']]
    chosen = max(good, key=lambda r: (r['metrics']['correct'], r['kind'] == 'mean'))['kind'] if good else 'lr'
    return dict(selected=chosen, baseline=baseline, candidates=rows,
                rule='No outer fold below LR; pooled gain >=0.1 percentage point; attention must beat mean; ties prefer mean; else LR',
                scope='Train OOF model selection, not unbiased test. Calls are disjoint; speakers may overlap.')


def develop(jobs, values, out, device):
    y = np.array([j['gender'] == 'M' for j in jobs], dtype=np.int64)
    ids = np.array([j['call_id'] for j in jobs]); plans = split_plan(y)
    ps = {k: np.zeros((len(y), 2), dtype=np.float64) for k in ('lr', 'mean', 'attention')}
    folds = np.full(len(y), -1); epochs = {k: [] for k in ('mean', 'attention')}
    measured_fold_seconds = []
    for k, (tr, test, inner, stop) in enumerate(plans):
        fold_path = out / 'development' / f'fold_{k}.pt'
        if fold_path.exists():
            result = load(fold_path)
            require(np.array_equal(result['ids'], ids[test]), 'OOF ID mismatch')
        else:
            fold_start = time.monotonic()
            subset = lambda ix: [values[i] for i in ix]
            p = {'lr': predict(fit_lr(subset(tr), y[tr]), subset(test), device)}; ep = {}
            for kind in ('mean', 'attention'):
                tune = train_head(subset(inner), y[inner], kind, out/'development'/f'fold_{k}_{kind}_tune.pt', device, subset(stop), y[stop])
                ep[kind] = tune['epochs']
                refit = train_head(subset(tr), y[tr], kind, out/'development'/f'fold_{k}_{kind}_refit.pt', device, epochs=ep[kind])
                p[kind] = predict(refit, subset(test), device)
            result = dict(ids=ids[test], predictions=p, epochs=ep)
            torch_save(fold_path, result)
            measured_fold_seconds.append(time.monotonic()-fold_start)
        for kind in ps:
            metrics(y[test], result['predictions'][kind])
            ps[kind][test] = result['predictions'][kind]
        for kind in epochs: epochs[kind].append(result['epochs'][kind])
        folds[test] = k
        print(f'Outer fold {k+1}/3 complete', flush=True)
        if measured_fold_seconds:
            print(f'Remaining fold ETA: {np.mean(measured_fold_seconds)*(2-k)/60:.1f} min (final refit/evaluation extra)', flush=True)
    require((folds >= 0).all(), 'OOF coverage')
    selection = select(y, ps, folds)
    selection['epoch_counts'] = epochs
    selection['final_epochs'] = int(np.median(epochs[selection['selected']])) if selection['selected'] != 'lr' else None
    torch_save(out/'development/oof.pt', dict(ids=ids, y=y, folds=folds, predictions=ps))
    return selection
''', encoding='utf-8')
print('heads.py' + ' 준비 완료')


In [ ]:
#@title 실험 실행 코드 저장
(WORK/'run.py').write_text(r'''"""L4 window weighting experiment: benchmark, extract Train, nested selection, locked evaluation."""
import argparse
import csv
import io
import json
import os
from pathlib import Path
import platform
import shutil
import time
import zipfile

import numpy as np
import torch

from features import (MODEL, REVISION, WINDOW, INPUT_SHA, Encoder, atomic, benchmark_jobs,
                      caller_samples, digest, extract, load_jobs, read, read_block,
                      read_partition, require, save, sha, waveform, write_block)
from heads import MAX_EPOCHS, PATIENCE, BATCH, develop, fit_lr, load, metrics, predict, train_head, torch_save


def environment(config):
    os.environ['HF_HOME'] = config['hf_cache']
    import scipy, sklearn, transformers, soundfile
    require(torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(), 'This experiment expects an L4 GPU')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(2)
    return dict(model=MODEL, revision=REVISION, input_sha256=INPUT_SHA,
                source={n: sha(Path(__file__).with_name(n)) for n in ('run.py', 'features.py', 'heads.py')},
                config=config, window_seconds=WINDOW, feature='last_hidden_state mean per balanced window',
                gpu=torch.cuda.get_device_name(), cuda=torch.version.cuda, precision='bf16 extraction / fp32 head',
                python=platform.python_version(), packages={k: v.__version__ for k, v in [('torch', torch), ('numpy', np), ('scipy', scipy), ('sklearn', sklearn), ('transformers', transformers), ('soundfile', soundfile)]},
                design=dict(folds=3, inner_stop_fraction=.15, max_epochs=MAX_EPOCHS, patience=PATIENCE,
                            batch=BATCH, head_lr=.001, weight_decay=.001, dropout=.1, prior_KL=.01,
                            candidates=['lr', 'mean', 'attention'], seed=42))


def benchmark(jobs, config, out, ih):
    path = out/'benchmark.json'
    if path.exists():
        report = read(path); require(report['identity'] == ih, 'Benchmark identity')
        print(json.dumps(report, indent=2)); return
    subset = benchmark_jobs(jobs)
    missing = [j for j in subset if not (Path(config['data_root'])/j['wav_relative_path']).is_file()]
    require(not missing, 'Audio DATA_ROOT incorrect or missing WAV: ' + (missing[0]['wav_relative_path'] if missing else ''))
    t = time.monotonic(); encoder = Encoder(); model_load = time.monotonic()-t
    # Warm up separately; never train the frozen backbone.
    encoder.encode(waveform(subset[len(subset)//2], config['data_root']))
    records = []
    for i, j in enumerate(subset):
        p = out/'benchmark_features'/(j['call_id']+'.npz')
        timing = p.with_suffix('.json')
        if p.exists() and timing.exists():
            read_block(p, [j], ih); r = read(timing)
            require(r['identity'] == ih, 'Benchmark timing identity')
        else:
            t = time.monotonic()
            value = encoder.encode(waveform(j, config['data_root']))
            write_block(p, [j], [value], ih)
            r = dict(identity=ih, call_id=j['call_id'], seconds=time.monotonic()-t, samples=caller_samples(j))
            save(timing, r)
        records.append(r)
        if (i+1) % 20 == 0: print(f'Benchmark: {i+1}/200', flush=True)
    ratio = sum(r['seconds'] for r in records) / sum(r['samples'] for r in records)
    estimates = {p: ratio*sum(caller_samples(j) for j in jobs if j['partition'] == p)/3600 for p in ('train', 'internal_validation')}
    report = dict(identity=ih, measured_calls=200, model_load_seconds=model_load,
                  measured_seconds=sum(r['seconds'] for r in records),
                  projected_extraction_hours=estimates,
                  total_extraction_hours=sum(estimates.values()),
                  note='Estimate, not deadline: includes WAV reads, BF16 forward and Drive writes. Drive throttling may dominate. Benchmark uses per-call files; full extraction writes 64-call blocks. Head CV time is extra and measured during development. Benchmark calls are reused.')
    save(path, report); print(json.dumps(report, indent=2), flush=True)


def export_results(out):
    with zipfile.ZipFile(out/'window_attention_results.zip.tmp', 'w', zipfile.ZIP_DEFLATED) as z:
        for name in ('identity.json', 'benchmark.json', 'selection.lock.json', 'final_models.json',
                     'validation_started.json', 'completion.json', 'REPORT.md', 'validation_predictions.csv', 'events.jsonl'):
            z.write(out/name, name)
        for path in sorted((out/'logs').glob('*.log')):
            z.write(path, 'logs/'+path.name)
    os.replace(out/'window_attention_results.zip.tmp', out/'window_attention_results.zip')


def final_fit(jobs, config, out, ih):
    lock = read(out/'selection.lock.json')
    require(lock['identity'] == ih, 'Selection identity mismatch')
    if (out/'final_models.json').exists():
        final = read(out/'final_models.json')
        require(final['selection_sha256'] == sha(out/'selection.lock.json') and final['checkpoint_sha256'] == sha(out/'model.pt'), 'Final checkpoint changed')
        return
    require(not (out/'validation_started.json').exists(), 'Cannot refit after evaluation starts')
    train, values = read_partition(jobs, out, ih, 'train')
    y = np.array([j['gender'] == 'M' for j in train], dtype=np.int64)
    lr = fit_lr(values, y)
    selected = lr if lock['selected'] == 'lr' else train_head(values, y, lock['selected'], out/'final_head_resume.pt', 'cuda', epochs=lock['final_epochs'])
    checkpoint = dict(identity=ih, selected=selected, baseline=lr, class_order=['F', 'M'],
                      model=MODEL, revision=REVISION, window_seconds=WINDOW,
                      selection_sha256=sha(out/'selection.lock.json'))
    torch_save(out/'model.pt', checkpoint)
    save(out/'final_models.json', dict(identity=ih, selection_sha256=checkpoint['selection_sha256'], checkpoint_sha256=sha(out/'model.pt')))


def evaluate(jobs, config, out, ih):
    final = read(out/'final_models.json'); lock = read(out/'selection.lock.json')
    require(final['identity'] == lock['identity'] == ih, 'Final identity changed')
    require(final['selection_sha256'] == sha(out/'selection.lock.json') and final['checkpoint_sha256'] == sha(out/'model.pt'), 'Selection/model hash mismatch')
    marker = dict(identity=ih, final_models_sha256=sha(out/'final_models.json'))
    if (out/'validation_started.json').exists(): require(read(out/'validation_started.json') == marker, 'Evaluation identity changed')
    else: save(out/'validation_started.json', marker)
    extract(jobs, config['data_root'], out, ih, 'internal_validation')
    val, values = read_partition(jobs, out, ih, 'internal_validation')
    y = np.array([j['gender'] == 'M' for j in val], dtype=np.int64)
    checkpoint = load(out/'model.pt')
    p = predict(checkpoint['selected'], values, 'cuda'); baseline = predict(checkpoint['baseline'], values, 'cuda')
    selected_metrics, base_metrics = metrics(y, p), metrics(y, baseline)
    slices = {}
    durations = np.array([caller_samples(j)/16000 for j in val])
    for name, mask in [('under_12s', durations < 12), ('12_to_36s', (durations >= 12) & (durations < 36)), ('36s_plus', durations >= 36)]:
        if mask.any():
            slices[name] = dict(n=int(mask.sum()), selected_accuracy=float((p[mask].argmax(1)==y[mask]).mean()), baseline_accuracy=float((baseline[mask].argmax(1)==y[mask]).mean()))
    new_ok, old_ok = p.argmax(1) == y, baseline.argmax(1) == y
    result = dict(status='complete', identity=ih, selected=lock['selected'], metrics=selected_metrics,
                  same_pipeline_LR=base_metrics, versus_LR=dict(corrected=int((new_ok & ~old_ok).sum()), regressed=int((~new_ok & old_ok).sum())),
                  duration_slices=slices, reached_99=selected_metrics['correct'] >= 5542,
                  historical_results=dict(wavlm_correct=5490, ensemble_correct=5497),
                  historical_baseline_count_reproduced=base_metrics['correct'] == 5490,
                  scope='Previously observed Internal Validation; not independent test. Official Validation unused. No speaker-disjoint guarantee. Historical comparisons are descriptive, not candidate selection.')
    s = io.StringIO(); w = csv.writer(s, lineterminator='\n')
    w.writerow(['call_id', 'gender', 'prediction', 'p_F', 'p_M', 'baseline_prediction', 'caller_seconds'])
    for j, probs, b, seconds in zip(val, p, baseline, durations):
        w.writerow([j['call_id'], j['gender'], 'FM'[int(probs.argmax())], *probs, 'FM'[int(b.argmax())], seconds])
    atomic(out/'validation_predictions.csv', s.getvalue().encode())
    report = f"# Frozen WavLM window weighting\n\nSelected: {lock['selected']}\n\nAccuracy: {selected_metrics['accuracy']:.6%} ({selected_metrics['correct']}/5597), wrong: {selected_metrics['wrong']}\n\nSame-pipeline LR: {base_metrics['accuracy']:.6%}\n\nHistorical ensemble: 98.213329% (5497/5597).\n\n{result['scope']}\n"
    atomic(out/'REPORT.md', report.encode()); save(out/'completion.json', result)
    print(report, flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('stage', choices=['benchmark', 'extract-train', 'develop', 'final-fit', 'evaluate'])
    parser.add_argument('--config', required=True)
    args = parser.parse_args(); config = read(args.config)
    # Import Transformers only after disabling unrelated optional vision/audio modules.
    import sys
    sys.modules['torchvision'] = None; sys.modules['librosa'] = None
    out = Path(config['output']); out.mkdir(parents=True, exist_ok=True)
    require(out.resolve() != Path(config['data_root']).resolve(), 'Separate output/data directories required')
    import fcntl
    mutex = open('/tmp/aigo-wavlm-window-gpu.lock', 'w')
    fcntl.flock(mutex, fcntl.LOCK_EX | fcntl.LOCK_NB)
    # Same VM lock only: one runtime per shared Drive output is required.
    jobs = load_jobs(config['input_json']); ident = environment(config); ih = digest(ident)
    if (out/'identity.json').exists(): require(read(out/'identity.json') == ident, 'Environment/source/config changed. Use a new output directory.')
    else:
        require(not any(p.name != 'logs' for p in out.iterdir()), 'Nonempty output without identity')
        save(out/'identity.json', ident)
    started = time.monotonic(); event = dict(stage=args.stage, identity=ih, started=time.time())
    try:
        if args.stage == 'benchmark':
            require(shutil.disk_usage(out).free > 4*2**30, 'Need 4 GiB free output space')
            benchmark(jobs, config, out, ih)
        else:
            require(read(out/'benchmark.json')['identity'] == ih, 'Benchmark first')
            if args.stage == 'extract-train':
                require(not (out/'selection.lock.json').exists(), 'Selection already locked; proceed to final-fit/evaluate')
                extract(jobs, config['data_root'], out, ih, 'train')
            elif args.stage == 'develop':
                if (out/'selection.lock.json').exists():
                    print(json.dumps(read(out/'selection.lock.json'), indent=2))
                else:
                    require(not (out/'validation_started.json').exists(), 'No development after evaluation')
                    train, values = read_partition(jobs, out, ih, 'train')
                    result = develop(train, values, out, 'cuda'); result['identity'] = ih
                    save(out/'selection.lock.json', result); print(json.dumps(result, indent=2), flush=True)
            elif args.stage == 'final-fit': final_fit(jobs, config, out, ih)
            elif args.stage == 'evaluate':
                if not (out/'completion.json').exists(): evaluate(jobs, config, out, ih)
                else: print((out/'REPORT.md').read_text())
        event.update(status='complete', session_seconds=time.monotonic()-started)
    except BaseException as exc:
        event.update(status='failed', error=repr(exc), session_seconds=time.monotonic()-started)
        raise
    finally:
        with (out/'events.jsonl').open('a') as f:
            f.write(json.dumps(event)+'\n'); f.flush(); os.fsync(f.fileno())
    if args.stage == 'evaluate': export_results(out)


if __name__ == '__main__': main()
''', encoding='utf-8')
print('run.py' + ' 준비 완료')


## 3. 별도 환경 준비
현재 Colab의 CUDA PyTorch를 재사용합니다. 새 venv에 CPU 처리·Transformers 패키지만 설치합니다.


In [ ]:
#@title 실행 환경 준비
assert (3, 10) <= sys.version_info[:2] < (3, 14), 'Python 3.10–3.13 런타임이 필요합니다.'
ENV = Path('/content/wavlm_window_env_v1')
PY = ENV/'bin/python'
(WORK/'requirements.txt').write_text('numpy==2.2.2\nscipy==1.15.3\nscikit-learn==1.6.1\nsoundfile==0.13.1\ntransformers==4.46.3\ntokenizers==0.20.3\nhuggingface-hub==0.31.4\nsafetensors==0.5.3\njoblib==1.4.2\nthreadpoolctl==3.6.0\n')
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(ENV)], check=True)
subprocess.run([sys.executable, '-m', 'pip', '--python', str(PY), 'install', '-q', '--ignore-installed', '-r', str(WORK/'requirements.txt')], check=True)
subprocess.run([str(PY), '-c', "import torch,numpy,scipy; assert torch.cuda.is_available(); print('PyTorch',torch.__version__, 'GPU',torch.cuda.get_device_name(0), 'NumPy',numpy.__version__)"] , check=True)
print('환경 준비 완료')


In [ ]:
#@title 실행 함수 준비 — 진행 출력과 로그 저장, Stop 시 자식 종료
import signal, time
def run(stage):
    logs = Path(OUTPUT)/'logs'
    logs.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ, HF_HOME=HF_CACHE, CUBLAS_WORKSPACE_CONFIG=':4096:8',
               TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')
    def kill_when_parent_dies():
        import ctypes
        parent = os.getppid()
        ctypes.CDLL(None).prctl(1, signal.SIGTERM)
        if os.getppid() != parent: os.kill(os.getpid(), signal.SIGTERM)
    p = subprocess.Popen([str(PY), '-u', str(WORK/'run.py'), stage, '--config', str(CONFIG)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        env=env, start_new_session=True, preexec_fn=kill_when_parent_dies)
    try:
        with (logs/(stage+'.log')).open('a') as log:
            for line in p.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
        if p.wait(): raise RuntimeError(f'{stage} 실패. 출력과 Drive logs 폴더를 확인하고 같은 셀로 재개하세요.')
    finally:
        if p.poll() is None:
            os.killpg(p.pid, signal.SIGTERM)
            try: p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(p.pid, signal.SIGKILL); p.wait()


## 4. 200통화 속도 측정
모델 로드/워밍업은 별도입니다. 음성 길이 분포를 대표하는 Train 200통화를 측정하고 특징을 저장합니다.
아래 출력의 `total_extraction_hours`는 전체 Train+Internal Validation 추출의 추정 시간입니다.
Drive 속도 변화로 초과할 수 있고, 작은 분류기 학습 시간은 별도입니다.


In [ ]:
run('benchmark')


## 5. 전체 실험
Train 특징 → 내부 early stopping을 포함한 3-fold 비교 → 후보 선택 잠금 → 전체 Train 재학습 → 최종 평가.

- **LR:** 기존 방식의 길이 가중 평균 + LR(C=0.1).
- **Mean:** 같은 평균 + 작은 분류기.
- **Attention:** 같은 분류기 + 학습한 구간 비중. dropout·weight decay·길이 가중치에서 과도하게 벗어나지 않도록 하는 규제를 사용.
- 후보는 LR보다 모든 fold에서 하락하지 않고 합계 +0.1%p 이상일 때 채택합니다.
- Attention은 Mean보다도 좋아야 채택합니다. 조건 미달이면 LR 유지.
- 후보 선택용 OOF 점수 자체는 독립 test 성능이 아닙니다. 화자 ID가 없어 동일 사람의 통화가 서로 다른 fold에 들어갈 가능성은 남아 있습니다.

이 셀은 여러 시간 실행될 수 있습니다. Stop 후 같은 셀로 재개하면 완료 단계는 보존합니다.
오류가 나면 다음 단계로 넘어가지 않습니다. 평가 결과를 본 뒤 후보/규칙을 바꾸지 마세요.


In [ ]:
out = Path(OUTPUT)
if not (out/'selection.lock.json').exists():
    run('extract-train')
    run('develop')
if not (out/'final_models.json').exists():
    run('final-fit')
run('evaluate')


## 6. 결과 다운로드
`model.pt`는 작은 분류기 및 정규화 설정입니다. frozen WavLM 가중치는 별도로 필요합니다.
이번 산출물은 실험용이며, 대회 제출용 새 음성 `inference.py`와 최종 CSV 포장은 별도입니다.
기존 앙상블 98.213329%는 과거 참고값으로만 표시하며 현재 후보 선택에 사용하지 않습니다.


In [ ]:
from google.colab import files
out = Path(OUTPUT)
result = json.loads((out/'completion.json').read_text())
assert result['status'] == 'complete'
print((out/'REPORT.md').read_text())
print('선택:', result['selected'], '| 99% 달성:', result['reached_99'])
print('모델:', out/'model.pt')
files.download(str(out/'window_attention_results.zip'))
